# ask 1: Supervised Machine Learning Models

I will train and compare two regression algorithms and two classification algorithms. The regression task predicts traffic volume. The classification task uses a proxy risk label

In [1]:
#Task 1: Supervised Machine Learning Models.
# Import required packages and configure the logging as suggested.

import logging
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

PART3_DIR = Path.cwd().parent
REPO_DIR = PART3_DIR.parent

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(
            PART3_DIR / "part3.log",
            encoding="utf-8"
        )
    ],
    force=True
)

logger = logging.getLogger(__name__)

logger.info("Supervised learning notebook is started.")
logger.info(
    "Package versions: NumPy %s, Pandas %s, scikit-learn %s.",
    np.__version__,
    pd.__version__,
    sklearn.__version__
)

2026-10-10 18:49:30,533 | INFO | __main__ | Supervised learning notebook is started.
2026-10-10 18:49:30,534 | INFO | __main__ | Package versions: NumPy 2.4.6, Pandas 2.3.3, scikit-learn 1.7.2.


## Load and check the data

Load the original traffic CSV and check that all required columns are present before processing it.

In [2]:
#Load the original traffic data and check the required columns.
input_path = (
    REPO_DIR
    / "part1_data_analytics"
    / "Metro_Interstate_Traffic_Volume.csv"
)

required_columns = [
    "holiday",
    "temp",
    "rain_1h",
    "snow_1h",
    "clouds_all",
    "weather_main",
    "weather_description",
    "date_time",
    "traffic_volume"
]

try:
    data = pd.read_csv(input_path, keep_default_na=False)

    missing_columns = [
        column for column in required_columns
        if column not in data.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Required columns are missing: {missing_columns}."
        )

    logger.info(
        "Raw data has been loaded. There are %s rows and %s columns.",
        data.shape[0],
        data.shape[1]
    )

except (OSError, ValueError, pd.errors.ParserError):
    logger.error("Raw data could not be loaded.", exc_info=True)
    raise




display(data.head())

2026-10-10 18:50:31,384 | INFO | __main__ | Raw data has been loaded. There are 48204 rows and 9 columns.


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,None,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,None,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,None,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,None,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,None,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


## Prepare the data

I would reuse the cleaning functions from Part 2. Removing 17 exact duplicates leaves 48187 records. Invalid temperature and rainfall readings are marked as missing for

In [3]:
#Reuse the Part 2 cleaning steps.

import sys

part2_path = REPO_DIR / "part2_python"

if str(part2_path) not in sys.path:
    sys.path.insert(0, str(part2_path))

from pipeline import (
    standardise_categories,
    clean_datetime,
    convert_numeric_columns,
    fill_missing_categories,
    clean_other_numbers,
    remove_duplicates
)

model_data = standardise_categories(data)
model_data = clean_datetime(model_data)
model_data = convert_numeric_columns(model_data)
model_data = fill_missing_categories(model_data)
model_data = clean_other_numbers(model_data)
model_data = remove_duplicates(model_data)

# Mark invalid readings 
invalid_temperature = (
    ~np.isfinite(model_data["temp"])
    | (model_data["temp"] <= 0)
)

invalid_rainfall = (
    ~np.isfinite(model_data["rain_1h"])
    | (model_data["rain_1h"] < 0)
    | (model_data["rain_1h"] > 9000)
)

for column, invalid in [
    ("temp", invalid_temperature),
    ("rain_1h", invalid_rainfall)
]:
    count = int(invalid.sum())
    model_data.loc[invalid, column] = np.nan

    if count > 0:
        logger.warning(
            "%s: marked %s invalid readings as missing..",
            column,
            count
        )

# Keep the records in time order.
model_data = (
    model_data.sort_values("date_time", kind="stable")
    .reset_index(drop=True)
)

logger.info(
    "Data preparation has been completed. There are %s rows and %s columns.",
    model_data.shape[0],
    model_data.shape[1]
)

display(model_data[["temp", "rain_1h"]].isna().sum())

2026-10-10 18:52:00,681 | INFO | pipeline | holiday: no text changes are required.
2026-10-10 18:52:00,725 | INFO | pipeline | weather_main: no text changes are required.
2026-10-10 18:52:00,762 | WARNING | pipeline | weather_description: changed 1730 rows to fix spaces and capital letters.
2026-10-10 18:52:00,808 | INFO | pipeline | Date check has passed. No missing or invalid dates.
2026-10-10 18:52:00,809 | INFO | pipeline | date_time is now in datetime format.
2026-10-10 18:52:00,817 | INFO | pipeline | temp: number conversion finished.
2026-10-10 18:52:00,819 | INFO | pipeline | rain_1h: number conversion finished.
2026-10-10 18:52:00,821 | INFO | pipeline | snow_1h: number conversion finished.
2026-10-10 18:52:00,823 | INFO | pipeline | clouds_all: number conversion finished.
2026-10-10 18:52:00,825 | INFO | pipeline | traffic_volume: number conversion finished.
2026-10-10 18:52:00,840 | INFO | pipeline | holiday: no missing category values.
2026-10-10 18:52:00,849 | INFO | pipel

temp       10
rain_1h     1
dtype: int64

## Split the data by time

I would like to use 75% of recorded timestamps for training, 15% for validation and 10% for testing. REeorcd with the same tomestamo stay together. This gives 35873 training rows, 7370 validation rows and 4944 test rows.

In [4]:
# Split the data by time

timestamps = (
    model_data["date_time"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

train_end = int(len(timestamps) * 0.75)
validation_end = int(len(timestamps) * 0.90)

validation_start = timestamps.iloc[train_end]
test_start = timestamps.iloc[validation_end]



train_data = model_data.loc[
    model_data["date_time"] < validation_start
].copy()


validation_data = model_data.loc[
    (model_data["date_time"] >= validation_start)
    & (model_data["date_time"] < test_start)
].copy()

test_data = model_data.loc[
    model_data["date_time"] >= test_start
].copy()

#Check that the sets are separate and contain all records.

assert (
    train_data["date_time"].max()
    < validation_data["date_time"].min()
)
assert (
    validation_data["date_time"].max()
    < test_data["date_time"].min()
)
assert (
    len(train_data) + len(validation_data) + len(test_data)
    == len(model_data)
)

split_summary = pd.DataFrame([
    {
        "set": name,
        "rows": len(frame),
        "timestamps": frame["date_time"].nunique(),
        "first_date": frame["date_time"].min(),
        "last_date": frame["date_time"].max()
    }
    for name, frame in [
        ("Training", train_data),
        ("Validation", validation_data),
        ("Test", test_data)
    ]
])

logger.info(
    "Time split has been completed. Training: %s, validation: %s, test: %s rows.",
    len(train_data),
    len(validation_data),
    len(test_data)
)

display(split_summary)

2026-10-10 18:57:14,310 | INFO | __main__ | Time split has been completed. Training: 35873, validation: 7370, test: 4944 rows.


,set,rows,timestamps,first_date,last_date
0,Training,35873,30431,2012-10-02 09:00:00,2017-08-02 23:00:00
1,Validation,7370,6086,2017-08-03 00:00:00,2018-04-14 15:00:00
2,Test,4944,4058,2018-04-14 16:00:00,2018-09-30 23:00:00


## Fill missing readings

I calculate monthly medians from training data only and apply them to all thethree sets. This prevents validation and test data from influencing the replacement values.

In [5]:
#Calculat monthly medians using training data only.
imputation_values = {}

for column in ["temp", "rain_1h"]:
    monthly_medians = train_data.groupby(
        train_data["date_time"].dt.month
    )[column].median()

    
    fallback_median = train_data[column].median()

    
    if pd.isna(fallback_median):
        raise ValueError(
            f"No valid training readings are available for {column}."
        )

    imputation_values[column] = {
        "monthly": monthly_medians,
        "fallback": fallback_median
    }

#Apply the same training medians to all three sets.
for set_name, frame in [
    ("Training", train_data),
    ("Validation", validation_data),
    ("Test", test_data)
]:
    
    for column in ["temp", "rain_1h"]:
        missing = frame[column].isna()
        count = int(missing.sum())

        replacements = (
            frame["date_time"].dt.month
            .map(imputation_values[column]["monthly"])
            .fillna(imputation_values[column]["fallback"])
        )


        
        frame.loc[missing, column] = replacements.loc[missing]

        if count > 0:
            logger.warning(
                "%s: filled %s missing %s readings using training medians.",
                set_name,
                count,
                column
            )

    assert not frame[["temp", "rain_1h"]].isna().any().any()


logger.info(
    "Imputation has been completed using values calculated from training data only."
)

display(pd.DataFrame({
    column: values["monthly"]
    for column, values in imputation_values.items()
}).rename_axis("month"))

2026-10-10 18:58:51,915 | WARNING | __main__ | Training: filled 10 missing temp readings using training medians.
2026-10-10 18:58:51,925 | WARNING | __main__ | Training: filled 1 missing rain_1h readings using training medians.
2026-10-10 18:58:51,950 | INFO | __main__ | Imputation has been completed using values calculated from training data only.


,temp,rain_1h
month,,
1,266.935,0.0
2,267.400,0.0
3,273.080,0.0
4,278.830,0.0
5,287.150,0.0
6,292.560,0.0
7,295.170,0.0
8,293.995,0.0
9,291.600,0.0


## Create time and holiday features

I have added hour, weekday, month, weekend and holiday features. Sine and cosine encode the repeating hour and weekday cycles.

In [6]:
#Create time features and a holiday flag for each set.
def add_model_features(frame):
    featured = frame.copy()

    
    featured["hour"] = featured["date_time"].dt.hour
    featured["day_of_week"] = featured["date_time"].dt.dayofweek
    featured["month"] = featured["date_time"].dt.month

    
    # Monday is 0. Saturday and Sunday are weekends.
    featured["is_weekend"] = (
        featured["day_of_week"] >= 5
    ).astype(int)


    featured["is_holiday"] = (
        ~featured["holiday"].isin(["None", "Unknown"])
    ).astype(int)

    
    # Encode the repeating hour and weekday cycles.
    # Use sine and cosine to keep them closer
    hour_angle = 2 * np.pi * featured["hour"] / 24
    weekday_angle = 2 * np.pi * featured["day_of_week"] / 7

    featured["hour_sin"] = np.sin(hour_angle)
    featured["hour_cos"] = np.cos(hour_angle)
    featured["weekday_sin"] = np.sin(weekday_angle)
    featured["weekday_cos"] = np.cos(weekday_angle)

    featured["has_rain"] = (
        featured["rain_1h"] > 0
    ).astype(int)

    featured["has_snow"] = (
        featured["snow_1h"] > 0
    ).astype(int)

    return featured



train_data = add_model_features(train_data)
validation_data = add_model_features(validation_data)
test_data = add_model_features(test_data)

logger.info(
    "Time features, cyclical encodings, holiday flag and weather indicators "
    "have been added to all three sets."
)




display(train_data[
    [
        "date_time",
        "hour",
        "day_of_week",
        "is_weekend",
        "is_holiday",
        "hour_sin",
        "hour_cos",
        "weekday_sin",
        "weekday_cos"
    ]
].head())



2026-10-10 19:00:25,269 | INFO | __main__ | Time features, cyclical encodings, holiday flag and weather indicators have been added to all three sets.


,date_time,hour,day_of_week,is_weekend,is_holiday,hour_sin,hour_cos,weekday_sin,weekday_cos
0,2012-10-02 09:00:00,9,1,0,0,7.071068e-01,-0.707107,0.781831,0.62349
1,2012-10-02 10:00:00,10,1,0,0,5.000000e-01,-0.866025,0.781831,0.62349
2,2012-10-02 11:00:00,11,1,0,0,2.588190e-01,-0.965926,0.781831,0.62349
3,2012-10-02 12:00:00,12,1,0,0,1.224647e-16,-1.000000,0.781831,0.62349
4,2012-10-02 13:00:00,13,1,0,0,-2.588190e-01,-0.965926,0.781831,0.62349


In [7]:
# Calculate congestion thresholds from training data .

q1, q2, q3 = train_data["traffic_volume"].quantile(
    [0.25, 0.50, 0.75]
).values


SEVERE_WEATHER = {"Snow", "Thunderstorm", "Squall"}
LOW_VISIBILITY_WEATHER = {"Fog", "Mist", "Haze", "Smoke"}

#Apply the same thresholdsto every set.
for set_name, frame in [
    ("Training", train_data),
    ("Validation", validation_data),
    ("Test", test_data)
]:
    frame["congestion_category"] = np.select(
        [
            frame["traffic_volume"] <= q1,
            frame["traffic_volume"] <= q2,
            frame["traffic_volume"] <= q3
        ],
        ["Low", "Medium", "High"],
        default="Severe"
    )

    frame["is_low_visibility"] = (
        frame["weather_main"].isin(LOW_VISIBILITY_WEATHER)
    ).astype(int)

    high_congestion = frame["congestion_category"].isin(
        ["High", "Severe"]
    )

    risky_weather = (
        frame["weather_main"].isin(SEVERE_WEATHER)
        | frame["is_low_visibility"].eq(1)
    )

    frame["high_risk"] = (
        high_congestion & risky_weather
    ).astype(int)

    logger.info(
        "%s proxy labels have been created. High-risk records: %s of %s.",
        set_name,
        int(frame["high_risk"].sum()),
        len(frame)
    )

display(pd.DataFrame({
    "boundary": ["25th percentile", "50th percentile", "75th percentile"],
    "traffic_volume": [q1, q2, q3]
}))

display(
    train_data.groupby(
        ["congestion_category", "high_risk"]
    ).size().rename("records").reset_index()
)

2026-10-10 19:01:51,873 | INFO | __main__ | Training proxy labels have been created. High-risk records: 3852 of 35873.
2026-10-10 19:01:51,884 | INFO | __main__ | Validation proxy labels have been created. High-risk records: 1138 of 7370.
2026-10-10 19:01:51,928 | INFO | __main__ | Test proxy labels have been created. High-risk records: 476 of 4944.


,boundary,traffic_volume
0,25th percentile,1175.0
1,50th percentile,3349.0
2,75th percentile,4923.0


,congestion_category,high_risk,records
0,High,0,7040
1,High,1,1927
2,Low,0,8973
3,Medium,0,8965
4,Severe,0,7043
5,Severe,1,1925


In [8]:
# Fefine the common features for classification and regression.

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = [
    "temp",
    "rain_1h",
    "snow_1h",
    "clouds_all",
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "is_holiday",
    "hour_sin",
    "hour_cos",
    "weekday_sin",
    "weekday_cos",
    "has_rain",
    "has_snow",
    "is_low_visibility"
]




categorical_features = ["weather_main"]
feature_columns = numeric_features + categorical_features

#Keep the target information out of the model inputs.
assert not {
    "traffic_volume",
    "congestion_category",
    "high_risk"
}.intersection(feature_columns)

X_train = train_data[feature_columns].copy()
X_validation = validation_data[feature_columns].copy()
X_test = test_data[feature_columns].copy()


# Regression target: recorded traffic volume.
y_train_regression = train_data["traffic_volume"]
y_validation_regression = validation_data["traffic_volume"]
y_test_regression = test_data["traffic_volume"]


# Classification target: the documented proxy label.
y_train_classification = train_data["high_risk"]
y_validation_classification = validation_data["high_risk"]
y_test_classification = test_data["high_risk"]

#Give each model its own preprocessing object.
def build_preprocessor():
    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                StandardScaler(),
                numeric_features
            ),
            (
                "weather",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                categorical_features
            )
        ],
        remainder="drop"
    )

logger.info(
    "Common model inputs have been prepared with %s numeric features "
    "and %s categorical feature.",
    len(numeric_features),
    len(categorical_features)
)


display(X_train.head())

2026-10-10 19:21:55,824 | INFO | __main__ | Common model inputs have been prepared with 16 numeric features and 1 categorical feature.


,temp,rain_1h,snow_1h,clouds_all,hour,day_of_week,month,is_weekend,is_holiday,hour_sin,hour_cos,weekday_sin,weekday_cos,has_rain,has_snow,is_low_visibility,weather_main
0,288.28,0.0,0.0,40,9,1,10,0,0,7.071068e-01,-0.707107,0.781831,0.62349,0,0,0,Clouds
1,289.36,0.0,0.0,75,10,1,10,0,0,5.000000e-01,-0.866025,0.781831,0.62349,0,0,0,Clouds
2,289.58,0.0,0.0,90,11,1,10,0,0,2.588190e-01,-0.965926,0.781831,0.62349,0,0,0,Clouds
3,290.13,0.0,0.0,90,12,1,10,0,0,1.224647e-16,-1.000000,0.781831,0.62349,0,0,0,Clouds
4,291.14,0.0,0.0,75,13,1,10,0,0,-2.588190e-01,-0.965926,0.781831,0.62349,0,0,0,Clouds


In [9]:
# Train and compare two traffic-volume regression models.

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

regression_algorithms = {
    "Linear Regression": LinearRegression(),
"Random Forest": RandomForestRegressor(
        n_estimators=100,
        max_depth=20,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
}



regression_models = {}
regression_results = []

for model_name, algorithm in regression_algorithms.items():
    model = Pipeline([
        ("preprocessor", build_preprocessor()),
        ("model", algorithm)
    ])

    logger.info("Training %s regression model.", model_name)

    # Fit preprocessing using training data only.
    model.fit(X_train, y_train_regression)

    validation_predictions = model.predict(X_validation)

    mae = mean_absolute_error(
        y_validation_regression,
        validation_predictions
    )
    r_squared = r2_score(
        y_validation_regression,
        validation_predictions
    )

    regression_models[model_name] = model
    regression_results.append({
        "model": model_name,
        "validation_MAE": mae,
        "validation_R_squared": r_squared
    })

    logger.info(
        "%s validation results: MAE %.2f and R-squared %.4f.",
        model_name,
        mae,
        r_squared
    )

regression_comparison = (
    pd.DataFrame(regression_results)
    .sort_values("validation_MAE")
    .reset_index(drop=True)
)


display(regression_comparison.round(4))

2026-10-10 19:23:34,968 | INFO | __main__ | Training Linear Regression regression model.
2026-10-10 19:23:35,461 | INFO | __main__ | Linear Regression validation results: MAE 826.19 and R-squared 0.7158.
2026-10-10 19:23:35,463 | INFO | __main__ | Training Random Forest regression model.
2026-10-10 19:23:42,093 | INFO | __main__ | Random Forest validation results: MAE 317.57 and R-squared 0.9256.


,model,validation_MAE,validation_R_squared
0,Random Forest,317.5692,0.9256
1,Linear Regression,826.1898,0.7158


In [10]:
#Train and compare two proxy-risk classification models.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


classification_algorithms = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

classification_models = {}
classification_results = []

for model_name, algorithm in classification_algorithms.items():
    model = Pipeline([
        ("preprocessor", build_preprocessor()),
        ("model", algorithm)
    ])

    logger.info("Training %s classification model.", model_name)

    model.fit(X_train, y_train_classification)

    #Class predictions use the default probability threshold of 0.5.
    
    predictions = model.predict(X_validation)
    probabilities = model.predict_proba(X_validation)[:, 1]

    results = {
        "model": model_name,
        "validation_accuracy": accuracy_score(
            y_validation_classification, predictions
        ),
        "validation_precision": precision_score(
            y_validation_classification, predictions,
            zero_division=0
        ),
        "validation_recall": recall_score(
            y_validation_classification, predictions,
            zero_division=0
        ),
        "validation_F1": f1_score(
            y_validation_classification, predictions,
            zero_division=0
        ),
        "validation_ROC_AUC": roc_auc_score(
            y_validation_classification, probabilities
        )
    }

    classification_models[model_name] = model
    classification_results.append(results)

    
    logger.info(
        "%s validation results: accuracy %.4f, precision %.4f, "
        "recall %.4f, F1 %.4f and ROC AUC %.4f.",
        model_name,
        results["validation_accuracy"],
        results["validation_precision"],
        results["validation_recall"],
        results["validation_F1"],
        results["validation_ROC_AUC"]
    )

classification_comparison = (
    pd.DataFrame(classification_results)
    .sort_values("validation_F1", ascending=False)
    .reset_index(drop=True)
)



display(classification_comparison.round(4))

2026-10-10 19:26:06,651 | INFO | __main__ | Training Logistic Regression classification model.
2026-10-10 19:26:07,033 | INFO | __main__ | Logistic Regression validation results: accuracy 0.9507, precision 0.7645, recall 0.9842, F1 0.8605 and ROC AUC 0.9895.
2026-10-10 19:26:07,034 | INFO | __main__ | Training Random Forest classification model.
2026-10-10 19:26:08,000 | INFO | __main__ | Random Forest validation results: accuracy 0.9769, precision 0.9047, recall 0.9508, F1 0.9272 and ROC AUC 0.9932.


,model,validation_accuracy,validation_precision,validation_recall,validation_F1,validation_ROC_AUC
0,Random Forest,0.9769,0.9047,0.9508,0.9272,0.9932
1,Logistic Regression,0.9507,0.7645,0.9842,0.8605,0.9895


In [11]:
#Select the models using validation results only.
best_regression_name = regression_comparison.iloc[0]["model"]
best_classification_name = classification_comparison.iloc[0]["model"]

best_regression_model = regression_models[best_regression_name]
best_classification_model = classification_models[
    best_classification_name
]

logger.info(
    "Selected regression model: %s. Selected classifier: %s.",
    best_regression_name,
    best_classification_name
)

#Evaluate the selected regression model on the held-out test set.
test_volume_predictions = best_regression_model.predict(X_test)

regression_test_results = pd.DataFrame([{
    "model": best_regression_name,
    "test_MAE": mean_absolute_error(
        y_test_regression,
        test_volume_predictions
    ),
    "test_R_squared": r2_score(
        y_test_regression,
        test_volume_predictions
    )
}])


# Evaluate the selected classifier without changing its settings.
test_risk_predictions = best_classification_model.predict(X_test)
test_risk_probabilities = (
    best_classification_model.predict_proba(X_test)[:, 1]
)



classification_test_results = pd.DataFrame([{
    "model": best_classification_name,
    "test_accuracy": accuracy_score(
        y_test_classification, test_risk_predictions
    ),
    "test_precision": precision_score(
        y_test_classification, test_risk_predictions,
        zero_division=0
    ),
    "test_recall": recall_score(
        y_test_classification, test_risk_predictions,
        zero_division=0
    ),
    "test_F1": f1_score(
        y_test_classification, test_risk_predictions,
        zero_division=0
    ),
    "test_ROC_AUC": roc_auc_score(
        y_test_classification, test_risk_probabilities
    )
}])

logger.info("Selected models have been evaluated on the held-out test set.")


display(regression_test_results.round(4))
display(classification_test_results.round(4))

2026-10-10 19:27:28,347 | INFO | __main__ | Selected regression model: Random Forest. Selected classifier: Random Forest.
2026-10-10 19:27:28,597 | INFO | __main__ | Selected models have been evaluated on the held-out test set.


,model,test_MAE,test_R_squared
0,Random Forest,250.8358,0.9448


,model,test_accuracy,test_precision,test_recall,test_F1,test_ROC_AUC
0,Random Forest,0.9875,0.9091,0.9664,0.9369,0.9982


## Prepare common model inputs

Both tasks use the same features. Traffic volume, congestion category and the proxy label are excluded from inputs. Each model fits its weather encoder and scaler using thetraining data .

In [12]:
#Save trained models, preprocessing values and evaluation results.
import joblib

models_dir = PART3_DIR / "models"
results_dir = PART3_DIR / "results"

models_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

models_to_save = {
    "linear_regression_v1": regression_models["Linear Regression"],
    "random_forest_regression_v1": regression_models["Random Forest"],
    "logistic_regression_v1": classification_models["Logistic Regression"],
    "random_forest_classification_v1": classification_models["Random Forest"]
}

for filename, model in models_to_save.items():
    model_path = models_dir / f"{filename}.joblib"
    joblib.dump(model, model_path, compress=3)

    logger.info(
        "Model has been saved to models/%s.joblib.",
        filename
    )


preparation_config = {
    "imputation_values": imputation_values,
    "feature_columns": feature_columns,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "congestion_thresholds": [float(q1), float(q2), float(q3)],
    "severe_weather": sorted(SEVERE_WEATHER),
    "low_visibility_weather": sorted(LOW_VISIBILITY_WEATHER),
    "selected_regression": best_regression_name,
    "selected_classification": best_classification_name
}


joblib.dump(
    preparation_config,
    models_dir / "preparation_config_v1.joblib"
)

result_tables = {
    "regression_validation": regression_comparison,
    "classification_validation": classification_comparison,
    "regression_test": regression_test_results,
    "classification_test": classification_test_results
}


for filename, table in result_tables.items():
    table.to_csv(
        results_dir / f"{filename}.csv",
        index=False
    )

    logger.info(
        "Evaluation results have been saved to results/%s.csv.",
        filename
    )



2026-10-10 19:28:57,122 | INFO | __main__ | Model has been saved to models/linear_regression_v1.joblib.
2026-10-10 19:29:00,341 | INFO | __main__ | Model has been saved to models/random_forest_regression_v1.joblib.
2026-10-10 19:29:00,348 | INFO | __main__ | Model has been saved to models/logistic_regression_v1.joblib.
2026-10-10 19:29:00,557 | INFO | __main__ | Model has been saved to models/random_forest_classification_v1.joblib.
2026-10-10 19:29:00,578 | INFO | __main__ | Evaluation results have been saved to results/regression_validation.csv.
2026-10-10 19:29:00,583 | INFO | __main__ | Evaluation results have been saved to results/classification_validation.csv.
2026-10-10 19:29:00,586 | INFO | __main__ | Evaluation results have been saved to results/regression_test.csv.
2026-10-10 19:29:00,591 | INFO | __main__ | Evaluation results have been saved to results/classification_test.csv.


In [13]:
#Check that saved models preserve their predictions.
sample_inputs = X_validation.head(20)

for filename, original_model in models_to_save.items():
    loaded_model = joblib.load(
        models_dir / f"{filename}.joblib"
    )

    original_predictions = original_model.predict(sample_inputs)
    loaded_predictions = loaded_model.predict(sample_inputs)

    np.testing.assert_allclose(
        original_predictions,
        loaded_predictions
    )

    
    if hasattr(original_model, "predict_proba"):
        np.testing.assert_allclose(
            original_model.predict_proba(sample_inputs),
            loaded_model.predict_proba(sample_inputs)
        )

    logger.info(
        "Reload check has passed for %s.",
        filename
    )

saved_config = joblib.load(
    models_dir / "preparation_config_v1.joblib"
)

assert saved_config["feature_columns"] == feature_columns

logger.info("All saved models and preparation settings have passed reload checks.")

2026-10-10 19:30:09,498 | INFO | __main__ | Reload check has passed for linear_regression_v1.
2026-10-10 19:30:10,749 | INFO | __main__ | Reload check has passed for random_forest_regression_v1.
2026-10-10 19:30:10,810 | INFO | __main__ | Reload check has passed for logistic_regression_v1.
2026-10-10 19:30:11,055 | INFO | __main__ | Reload check has passed for random_forest_classification_v1.
2026-10-10 19:30:11,070 | INFO | __main__ | All saved models and preparation settings have passed reload checks.
